### 1 - importacoes

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2 - Importacao do dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("symptom_based_medicine_recommendation_dataset.csv")
df.head()

## 3- Analise exploratoria

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['disease'].value_counts()

## 4 - Vizualizacao da distribuicao


In [ ]:
plt.figure(figsize=(14,6))
df['disease'].value_counts().plot(kind='bar')

plt.xticks(rotation=45, ha='right')
plt.title("Distribuição das Doenças")

plt.tight_layout()
plt.show()

## 5 - Pre-processamento

In [ ]:
le = LabelEncoder()
df['disease'] = le.fit_transform(df['disease'])

In [ ]:
X = df.drop([  #removi colunas de string pra treinamento dos modelos
    'disease',
    'name',
    'recommended_medicine',
    'secondary_medicine',
    'reaction_name'
], axis=1)

y = df['disease']

X = pd.get_dummies(X) #converti os categoricos em numericos pra evitar de quebrar o codigo, assim eu pude treinar os modelos mesmo com dados em string

## 6 - Divisão treino/teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 7 - Treinamento dos Modelos

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

In [ ]:
from sklearn.preprocessing import StandardScaler # fiz isso pra subir a acuracia do KNN, saiu de 0,04 pra 0.895

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

## 8 - Avaliacao dos modelos

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

models = {
    "Random Forest": rf,
    "Decision Tree": dt,
    "KNN": knn
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\nModelo: {name}")
    print("Acurácia:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

### 9- Análise de Acertos do Modelo

In [ ]:
from sklearn.metrics import confusion_matrix    #Essa tabela mostra onde o modelo acerta e onde ele erra.
import seaborn as sns                           #A diagonal principal representa os acertos, e como podemos ver, a maioria dos valores está concentrada nela, indicando alto desempenho.
import matplotlib.pyplot as plt                 #Os valores fora da diagonal representam erros, mostrando que algumas doenças com sintomas semelhantes podem ser confundidas.

y_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Análise de Acertos do Modelo - Random Forest")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.show()

## 10 - otimizacao

In [ ]:
from sklearn.model_selection import GridSearchCV #Foi realizada uma otimização de hiperparâmetros utilizando GridSearchCV.
from sklearn.ensemble import RandomForestClassifier #O melhor modelo encontrado utilizou 100 árvores e profundidade máxima ilimitada,
                                                     #apresentando o melhor desempenho na validação cruzada.
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Melhores parâmetros:", grid.best_params_)

## 11 - teste de predicao

In [ ]:
sample = X.iloc[0].values.reshape(1, -1)
prediction = best_model.predict(sample)

print("Doença prevista:", le.inverse_transform(prediction)) #optei por deixar o nome original da doenca do .csv

##